# 계층형 청킹 실습 노트북

이 노트북은 **마크다운으로 변환된 법령 문서**를 대상으로  
`문서 제목 → 장 → 절 → 조` 구조를 인식하여 JSON 청크를 만드는 실습용입니다.

## 실습 목표

최종적으로 다음 형태의 JSON 리스트를 만드는 것이 목표입니다.

```json
{
  "document_title": "...",
  "chapter": "...",
  "section": "...",
  "article_no": "...",
  "article_title": "...",
  "content": "...",
  "source_path": "..."
}
```

주의할 점:

- 조문 하나를 하나의 청크로 만든다.
- `제10조`와 `제10조의2`는 서로 다른 청크로 처리한다.
- 본문은 다음 조문이 나오기 전까지 포함한다.
- 장과 절 정보는 metadata처럼 유지한다.

## 1. 파일 경로 설정

같은 폴더에 마크다운 파일을 넣고 파일명을 수정하세요.

In [ ]:
from pathlib import Path
import re
import json

# TODO: 실습 파일명을 본인 파일명으로 바꾸세요.
INPUT_PATH = Path("민원처리법_hwpx변환.md")

# 결과 저장 파일명
OUTPUT_PATH = Path("chunks.json")

print("입력 파일:", INPUT_PATH)
print("파일 존재 여부:", INPUT_PATH.exists())

## 2. 문서 일부 확인

먼저 문서의 구조를 직접 확인합니다.  
`#`, `##`, `###`, `**제n조...**` 패턴이 어떻게 나오는지 살펴보세요.

In [ ]:
text = INPUT_PATH.read_text(encoding="utf-8")

lines = text.splitlines()

# 처음 40줄만 확인
for i, line in enumerate(lines[:40], start=1):
    print(f"{i:03d}: {line}")

## 3. 구조 패턴 찾기

아래 셀을 실행해서 문서 안에서 제목, 장, 절, 조문처럼 보이는 줄을 찾아봅니다.  
이 단계에서는 아직 정규식을 완성하지 않아도 됩니다.

In [ ]:
for i, line in enumerate(lines, start=1):
    stripped = line.strip()

    if stripped.startswith("#"):
        print(f"{i:03d}: {stripped}")
    elif stripped.startswith("**제") and "조" in stripped:
        print(f"{i:03d}: {stripped[:120]}")

## 4. 정규식 설계

아래 TODO를 채우세요.

힌트:

- 제목: `# 문서명`
- 장: `## 제1장 총칙`
- 절: `### 제1절 민원의 신청 및 접수 등`
- 조문: `**제10조(제목)** 본문...`
- `제10조의2` 처리를 위해 `(?:의\d+)?` 같은 선택 패턴이 필요할 수 있습니다.

In [ ]:
# TODO 1: 문서 제목 패턴
title_pattern = re.compile(r"^#\s+(.+)$")

# TODO 2: 장 패턴
chapter_pattern = re.compile(r"^##\s+(제\d+장\s+.+)$")

# TODO 3: 절 패턴
section_pattern = re.compile(r"^###\s+(제\d+절\s+.+)$")

# TODO 4: 조문 패턴
# 목표:
# - 제10조
# - 제10조의2
# - 괄호 안 제목
# - 조문 제목 뒤에 바로 이어지는 본문
# 을 분리할 수 있어야 합니다.
article_pattern = re.compile(
    r"^\*\*"
    r"(제\d+조(?:의\d+)?)"
    r"(?:\((.*?)\))?"
    r"\*\*"
    r"\s*(.*)$"
)

print("정규식 준비 완료")

## 5. 정규식 테스트

몇 개의 샘플 줄에 대해 정규식이 제대로 작동하는지 확인하세요.  
특히 `제n조의2` 형태가 잘 분리되는지 확인합니다.

In [ ]:
sample_lines = [
    "# 문서 제목 예시",
    "## 제1장 총칙",
    "### 제1절 민원의 신청 및 접수 등",
    "**제10조(불필요한 서류 요구의 금지)** ① 본문이 바로 이어지는 경우",
    "**제10조의2(민원인의 요구에 의한 본인정보 공동이용)** ① 본문이 이어지는 경우",
    "**제45조** 삭제 <2022. 1. 11.>",
]

for s in sample_lines:
    print("\n원문:", s)

    for name, pattern in [
        ("title", title_pattern),
        ("chapter", chapter_pattern),
        ("section", section_pattern),
        ("article", article_pattern),
    ]:
        m = pattern.match(s)
        if m:
            print(name, "=>", m.groups())

## 6. 현재 상태 저장 방식 설계

한 줄씩 읽으면서 다음 상태를 관리합니다.

- `document_title`
- `current_chapter`
- `current_section`
- `current_article`

새로운 조문을 만나면:

1. 이전 조문이 있으면 먼저 저장한다.
2. 새 조문의 번호와 제목을 기록한다.
3. 이후 줄들을 새 조문의 본문으로 모은다.

In [ ]:
def build_source_path(document_title, chapter, section, article_no, article_title):
    """출처 경로 문자열을 만든다."""

    # TODO: article_title이 있으면 제n조(제목) 형태로 만들기
    if article_title:
        article_label = f"{article_no}({article_title})"
    else:
        article_label = article_no

    parts = [document_title, chapter, section, article_label]

    # None 또는 빈 문자열 제거
    return " > ".join([p for p in parts if p])

## 7. 조문 저장 함수 만들기

`current_article`에 모아둔 본문을 하나의 청크로 저장합니다.

주의:

- 본문이 비어 있으면 저장하지 않는다.
- `content_lines`는 줄 리스트이므로 `\n`으로 합친다.
- `source_path`를 함께 만든다.

In [ ]:
def save_current_article(chunks, current_article):
    """현재 조문을 chunks 리스트에 저장한다."""

    if current_article is None:
        return

    content = "\n".join(current_article["content_lines"]).strip()

    if not content:
        return

    source_path = build_source_path(
        document_title=current_article["document_title"],
        chapter=current_article["chapter"],
        section=current_article["section"],
        article_no=current_article["article_no"],
        article_title=current_article["article_title"],
    )

    chunk = {
        "chunk_id": len(chunks) + 1,
        "document_title": current_article["document_title"],
        "chapter": current_article["chapter"],
        "section": current_article["section"],
        "article_no": current_article["article_no"],
        "article_title": current_article["article_title"],
        "content": content,
        "source_path": source_path,
    }

    chunks.append(chunk)

## 8. 계층형 청킹 함수 구현

아래 함수가 핵심입니다.

흐름:

1. 문서를 줄 단위로 순회한다.
2. 제목을 만나면 `document_title` 갱신
3. 장을 만나면 `current_chapter` 갱신
4. 절을 만나면 `current_section` 갱신
5. 조문을 만나면 이전 조문 저장 후 새 조문 시작
6. 일반 줄은 현재 조문의 본문에 추가
7. 반복문 종료 후 마지막 조문 저장

In [ ]:
def hierarchical_chunk_markdown(md_text):
    document_title = None
    current_chapter = None
    current_section = None
    current_article = None

    chunks = []

    for line in md_text.splitlines():
        stripped = line.strip()

        # 빈 줄 처리
        if not stripped:
            if current_article is not None:
                current_article["content_lines"].append("")
            continue

        # 1. 문서 제목
        m = title_pattern.match(stripped)
        if m and document_title is None:
            document_title = m.group(1).strip()
            continue

        # 2. 장
        m = chapter_pattern.match(stripped)
        if m:
            save_current_article(chunks, current_article)
            current_article = None

            current_chapter = m.group(1).strip()
            current_section = None
            continue

        # 3. 절
        m = section_pattern.match(stripped)
        if m:
            save_current_article(chunks, current_article)
            current_article = None

            current_section = m.group(1).strip()
            continue

        # 4. 조문
        m = article_pattern.match(stripped)
        if m:
            save_current_article(chunks, current_article)

            article_no = m.group(1).strip()
            article_title = m.group(2).strip() if m.group(2) else ""
            inline_content = m.group(3).strip() if m.group(3) else ""

            current_article = {
                "document_title": document_title,
                "chapter": current_chapter,
                "section": current_section,
                "article_no": article_no,
                "article_title": article_title,
                "content_lines": [],
            }

            if inline_content:
                current_article["content_lines"].append(inline_content)

            continue

        # 5. 일반 본문
        if current_article is not None:
            current_article["content_lines"].append(stripped)

    # 마지막 조문 저장
    save_current_article(chunks, current_article)

    return chunks

## 9. 청킹 실행

생성된 청크 개수를 확인합니다.

In [ ]:
chunks = hierarchical_chunk_markdown(text)

print("생성된 청크 수:", len(chunks))

# 앞의 3개만 확인
for chunk in chunks[:3]:
    print("-" * 80)
    print("chunk_id:", chunk["chunk_id"])
    print("source_path:", chunk["source_path"])
    print("content preview:", chunk["content"][:200])

## 10. 특정 조문 검증

`제10조`와 `제10조의2`가 서로 다른 청크로 분리되었는지 확인합니다.

In [ ]:
target_articles = ["제10조", "제10조의2"]

for chunk in chunks:
    if chunk["article_no"] in target_articles:
        print("=" * 80)
        print("chunk_id:", chunk["chunk_id"])
        print("article_no:", chunk["article_no"])
        print("article_title:", chunk["article_title"])
        print("source_path:", chunk["source_path"])
        print("content preview:")
        print(chunk["content"][:500])

## 11. JSON 저장

`chunks.json`으로 저장합니다.

In [ ]:
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("저장 완료:", OUTPUT_PATH)

## 12. Vector DB 적재용 구조로 변환

Vector DB에는 보통 다음처럼 분리해서 넣습니다.

- `pageContent`: 임베딩할 본문
- `metadata`: 출처, 장, 절, 조문 정보

JSON 전체를 통째로 임베딩하지 않는 것이 좋습니다.

In [ ]:
vector_docs = []

for chunk in chunks:
    page_content = f"""문서명: {chunk['document_title']}
출처: {chunk['source_path']}

{chunk['content']}
""".strip()

    metadata = {
        "document_title": chunk["document_title"],
        "chapter": chunk["chapter"],
        "section": chunk["section"],
        "article_no": chunk["article_no"],
        "article_title": chunk["article_title"],
        "source_path": chunk["source_path"],
    }

    vector_docs.append({
        "pageContent": page_content,
        "metadata": metadata,
    })

print("Vector DB 적재용 문서 수:", len(vector_docs))
print(json.dumps(vector_docs[0], ensure_ascii=False, indent=2)[:1000])